In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules['pyBigWig'] = MagicMock()
import pandas as pd
import numpy as np
import torch
import gpn.model
from transformers import AutoModel, AutoModelForMaskedLM
from gpn.data import GenomeMSA  # should work now

/blue/egn6933/share/apatil2/conda/envs/pathogenecity/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
coding = pd.read_csv('/home/apatil2/Disease_risk_modeling/filtered_data_link/clinvar_20240805.missense_matched.txt',
                sep = '\t',
                low_memory = False)

In [ ]:
# msa_path = "/home/apatil2/Disease_risk_modeling/filtered_data_link/genomeMSA/89.zarr.zip"

# genome_msa = GenomeMSA(msa_path)
msa_path = "/blue/egn6933/apatil2/pz/89.zarr"  # ← plain folder
genome_msa = GenomeMSA(msa_path)


In [ ]:
# model_path = 'songlab/gpn-msa-sapiens'
# model = AutoModelForMaskedLM.from_pretrained(model_path).to(device)
# model.eval();

In [ ]:
coding.tail()

In [ ]:
def get_genome_embedding(chrom, pos_start, pos_end):
    
    msa = genome_msa.get_msa(str(chrom), pos_start, pos_end, strand="+", tokenize=True)
    msa = torch.tensor(np.expand_dims(msa, 0).astype(np.int64))
    
    #separating human from rest of species
    input_ids, aux_features = msa[:, :, 0], msa[:, :, 1:]
    input_ids.shape, aux_features.shape
    
    #pushing data to gpu
    input_ids = input_ids.to(device)
    aux_features = aux_features.to(device)
    
    #loading the model
    model = AutoModel.from_pretrained(model_path).to(device)
    model.eval()
    
    with torch.no_grad():
        embeddings = model(input_ids=input_ids, aux_features=aux_features).last_hidden_state
        
        return embeddings

In [20]:
e = get_genome_embedding(2, coding['POS'][0], coding['POS'][0]+128)
e1 = get_genome_embedding(2, coding['POS'][0]-64, coding['POS'][0]+64)

Loading weights: 100%|██████████| 193/193 [00:00<00:00, 2845.54it/s, Materializing param=encoder.layer.11.output.dense.weight]              
GPNRoFormerModel LOAD REPORT from: songlab/gpn-msa-sapiens
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 193/193 [00:00<00:00, 3013.77it/s, Materializing param=encoder.layer.11.output.dense.weight]              
GPNRoFormerModel LOAD REPORT from: songlab

In [21]:
e.shape
e= e.view(128,768)
e1 = e1.view(128, 768)

In [26]:
e2 = get_genome_embedding(2, coding['POS'][1], coding['POS'][1]+128)
e2 = e2.view(128, 768)
e3 = get_genome_embedding(2, coding['POS'][1]-64, coding['POS'][1]+64)
e3 = e3.view(128, 768)

Loading weights: 100%|██████████| 193/193 [00:00<00:00, 3086.42it/s, Materializing param=encoder.layer.11.output.dense.weight]              
GPNRoFormerModel LOAD REPORT from: songlab/gpn-msa-sapiens
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 193/193 [00:00<00:00, 3269.44it/s, Materializing param=encoder.layer.11.output.dense.weight]              
GPNRoFormerModel LOAD REPORT from: songlab

In [22]:
import torch.nn.functional as F
sim = F.cosine_similarity(e[0], e1[64], dim=0)
print(sim)

tensor(0.1370, device='cuda:0')


In [27]:
sim1 = F.cosine_similarity(e2[0], e3[64], dim=0)
print(sim1)

tensor(0.2337, device='cuda:0')


In [4]:
# from huggingface_hub import hf_hub_download

# zip_path = hf_hub_download(
#     repo_id="songlab/multiz100way",
#     filename="89.zarr.zip",
#     repo_type="dataset",
#     local_dir="/blue/egn6933/apatil2/",  
#     local_dir_use_symlinks=False,
#     force_download=True
# )
# print("Saved to:", zip_path)   # should be ~36 GB

In [5]:
# import zipfile
# p = "/home/apatil2/Disease_risk_modeling/filtered_data_link/genomeMSA/89.zarr.zip"
# with zipfile.ZipFile(p) as z:
#     print("Files in zip:", len(z.namelist()))
#     print(z.namelist()[:30])